In [4]:
import io
import sys
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import get_girder_client

In [5]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    rows = []
    limit = 50
    offset = 0
    with ThreadPoolExecutor(max_workers=8) as executor:
        while True:
            batch = gc.get(
                "aimdl/datafiles",
                parameters={
                    "limit": limit,
                    "offset": offset,
                    "dataType": "pdv_alpss_results",
                },
            )
            if not batch:
                break
            futures = [executor.submit(fetch_and_parse, gc, item) for item in batch]
            for future in as_completed(futures):
                rows.append(future.result())
            if len(batch) < limit:
                break
            offset += limit

    df = pd.DataFrame(rows)
    df = coerce_types(df)

/var/folders/yp/l_t6f1mn0yqclvv5qdshg2xr0000gn/T/ipykernel_24511/3324275922.py:54: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], errors="coerce")


In [8]:
df.head()

,Date,File Name,Velocity OK,Spall OK,Uncertainty OK,Error Message,Run Time,Velocity at Max Compression,Time at Max Compression,Velocity at Max Tension,...,HEL Detected,HEL Strength (GPa),HEL Uncertainty (GPa),HEL Free Surface Velocity (m/s),HEL Time Detection (ns),HEL Consecutive Points,HEL Segment Duration (ns),HEL Strain Rate,igsn,itemId
0,2026-03-17 21:18:00,C1--20251022--00002.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.864054,760.198712,0.000001,-146.270060,...,False,NaN,NaN,-7.376598,968.859375,12,0.085938,NaN,JHAMAB00021-16,69b9c5134d9c815b961aa2ec
1,2026-03-17 21:17:00,C1--20251022--00009.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.848154,1092.321038,0.000001,-123.587673,...,False,NaN,NaN,-3.337541,980.039062,101,0.781250,NaN,JHAMAB00021-16,69b9c4ea4e69e623f00204be
2,2026-03-17 21:17:00,C1--20251022--00010.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.846242,1094.988503,0.000001,124.939287,...,False,NaN,NaN,-8.952012,1063.843750,10,0.070313,NaN,JHAMAB00021-16,69b9c4e54d9c815b961aa2cd
3,2026-03-17 21:17:00,C1--20251022--00007.csv,True,True,True,hel: rejected - HEL detected velocity < thresh...,0 days 00:00:00.846463,1033.238230,0.000001,-4.953949,...,False,NaN,NaN,-1.573894,969.093750,70,0.539062,NaN,JHAMAB00021-16,69b9c4f67590bb7e1ed223aa
4,2026-03-17 21:17:00,C1--20251022--00004.csv,True,True,True,,0 days 00:00:00.845821,863.717350,0.000001,14.797900,...,True,0.201165,0.030222,-10.058228,1006.328125,11,0.078125,-0.000277,JHAMAB00021-16,69b9c5087590bb7e1ed223e3
